# TIM x Sapienza Machine Unlearning Hackathon

    Questo notebook prepara i dati della challenge, valuta il modello originale,
    costruisce baseline confrontabili e lancia la pipeline finale di machine
    unlearning.

## 1. Obiettivo del progetto

    Vogliamo produrre una submission con `model_artifact`,
    `execution_time.txt` e `validation_ids.csv`. Il modello finale deve
    dimenticare gli utenti del forget set senza degradare troppo la Precision@10.

    L'evaluator completo della MIA ufficiale non e' pubblico: per scegliere le
    configurazioni usiamo quindi una proxy locale dichiarata, non una replica
    della metrica nascosta.

## 2. Impostazioni, librerie e riproducibilita'

    Importiamo le librerie, impostiamo seed e percorsi, e rendiamo il notebook
    eseguibile anche in Colab. Il batch size di training non viene confuso con
    quello di inferenza: lo leggiamo piu' avanti dagli iperparametri disponibili.

In [ ]:
from copy import deepcopy
from pathlib import Path
import os
import pickle
import subprocess

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 92
VAL_SIZE = 0.11
EVAL_BATCH_SIZE = 1024
ID_COL, TARGET_PREFIX = "user_id", "target__"
REPO_URL = "https://github.com/michelepezza99/machine-unlearning-hackathon-sapienza.git"
COLAB_DIR = Path("/content/machine-unlearning-hackathon-sapienza")

if Path("/content").exists() and not Path("data").exists():
    cmd = ["git", "-C", str(COLAB_DIR), "pull", "--ff-only"] if COLAB_DIR.exists() else ["git", "clone", REPO_URL, str(COLAB_DIR)]
    subprocess.run(cmd, check=True)
    os.chdir(COLAB_DIR)

from utils.model import DynamicMLP

data_dir, output_dir = Path("data"), Path("outputs")
output_dir.mkdir(exist_ok=True)
train_files = sorted(data_dir.glob("*c000.csv"))
forget_file, model_file = data_dir / "forget_data.csv", data_dir / "model_artifact"

if not train_files:
    raise FileNotFoundError("Nessun file di training trovato in data/.")
for path in (forget_file, model_file):
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")

train_df = pd.concat((pd.read_csv(p, sep=";") for p in train_files), ignore_index=True)
forget_df = pd.read_csv(forget_file)

if ID_COL not in train_df or ID_COL not in forget_df:
    raise ValueError(f"Colonna '{ID_COL}' mancante.")
if set(train_df.columns) != set(forget_df.columns):
    raise ValueError("Training e forget set hanno schemi diversi.")
if train_df[ID_COL].duplicated().any() or forget_df[ID_COL].duplicated().any():
    raise ValueError("user_id duplicati: serve uno split per utenti unici.")

forget_df = forget_df.loc[:, train_df.columns]
target_cols = [c for c in train_df.columns if c.lower().startswith(TARGET_PREFIX.lower())]
feature_cols = [c for c in train_df.columns if c != ID_COL and c not in target_cols]
if not target_cols:
    raise ValueError(f"Nessuna target con prefisso '{TARGET_PREFIX}'.")

train_ids, forget_ids = set(train_df[ID_COL]), set(forget_df[ID_COL])
missing_forget_ids = forget_ids - train_ids
if missing_forget_ids:
    raise ValueError(f"{len(missing_forget_ids)} forget user_id non presenti nel training.")

retain_df = train_df.loc[~train_df[ID_COL].isin(forget_ids)].copy()
retain_train_df, validation_df = train_test_split(retain_df, test_size=VAL_SIZE, random_state=SEED, shuffle=True)
retain_train_df, validation_df = retain_train_df.copy(), validation_df.copy()

if set(retain_train_df[ID_COL]) & set(validation_df[ID_COL]) or retain_df[ID_COL].isin(forget_ids).any():
    raise RuntimeError("Split non coerente: overlap tra insiemi.")

validation_ids_file = output_dir / "validation_ids.csv"
validation_df[[ID_COL]].to_csv(validation_ids_file, index=False)

pd.DataFrame({
    "dataset": ["full_training", "retain_train", "validation", "forget"],
    "rows": [len(train_df), len(retain_train_df), len(validation_df), len(forget_df)],
})

## 3. Caricamento e verifica dei dati

    La cella precedente carica le partizioni CSV, il forget set e identifica
    `feature_cols` e `target_cols`. I controlli su schema e duplicati servono a
    evitare split incoerenti basati su righe quando il problema e' per utente.

## 4. Costruzione di retain, validation e forget set

    Separiamo `retain_df` in `retain_train_df` e `validation_df`. La validation
    resta fuori dal training per selezionare checkpoint e misurare utility, ma
    non la trattiamo come vero non-member set del modello originale: TIM potrebbe
    averla usata nel training iniziale.

## 5. Preprocessing di feature e target

    Replichiamo il preprocessing del repository: feature convertite in numeriche
    e valori mancanti/non validi sostituiti con zero. Per le target siamo piu'
    conservativi: non imputiamo automaticamente per non alterare la ground truth.

    ## 6. Caricamento del modello originale

    Ricostruiamo `DynamicMLP` dall'artifact e verifichiamo compatibilita' tra
    numero di feature, target e architettura. Se l'artifact non contiene l'ordine
    delle colonne, documentiamo che l'ordine CSV e' l'unico riferimento.

In [ ]:
def prepare_features(df, cols):
    values = df.loc[:, cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    replaced = int(values.isna().sum().sum())
    X = values.fillna(0.0).to_numpy(dtype=np.float32)
    if not np.isfinite(X).all():
        raise ValueError("Feature preprocessate non finite.")
    return X, replaced


def prepare_targets(df, cols):
    y = df.loc[:, cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if y.isna().any().any():
        raise ValueError("Target mancanti/non numeriche: non vengono imputate.")
    y = y.to_numpy(dtype=np.float32)
    if not np.isin(y, [0.0, 1.0]).all():
        raise ValueError("Le target devono essere binarie.")
    return y


X_retain, repl_retain = prepare_features(retain_train_df, feature_cols)
X_validation, repl_validation = prepare_features(validation_df, feature_cols)
X_forget, repl_forget = prepare_features(forget_df, feature_cols)
y_retain, y_validation, y_forget = [prepare_targets(df, target_cols) for df in (retain_train_df, validation_df, forget_df)]

with open(model_file, "rb") as f:
    model_payload = pickle.load(f)

required = {"state_dict", "architecture", "best_hyperparameters", "model_class_source"}
missing = required - set(model_payload)
if missing:
    raise KeyError(f"Chiavi mancanti nel model_artifact: {sorted(missing)}")

architecture = model_payload["architecture"]
if len(feature_cols) != architecture["input_dim"] or len(target_cols) != architecture["num_outputs"]:
    raise ValueError(f"Dataset/model incompatibili: {len(feature_cols)} feature, {len(target_cols)} target, arch={architecture}")
if "feature_cols" in model_payload and list(model_payload["feature_cols"]) != feature_cols:
    raise ValueError("Ordine feature diverso da quello del modello.")
if "target_cols" in model_payload and list(model_payload["target_cols"]) != target_cols:
    raise ValueError("Ordine target diverso da quello del modello.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DynamicMLP(architecture["input_dim"], architecture["hidden_layers"], architecture["num_outputs"])
model.load_state_dict(model_payload["state_dict"])
model = model.to(device).eval()
original_state_dict = deepcopy(model.state_dict())

print("Feature sostituite con 0:", {"retain": repl_retain, "validation": repl_validation, "forget": repl_forget})
print("Shapes:", X_retain.shape, X_validation.shape, X_forget.shape)

## 7. Metriche di valutazione

    Usiamo Precision@10 per l'utilita' del ranking multilabel e BCE media sui
    logit per una diagnostica probabilistica. Le stesse definizioni guidano le
    baseline; la pipeline finale usa implementazioni equivalenti nel modulo
    `final_unlearning_pipeline.py`.

    ## 8. Baseline del modello originale

    Valutiamo il modello fornito senza modificarlo. Questa baseline definisce il
    riferimento di utility che i metodi di unlearning dovrebbero preservare.

In [ ]:
def build_model(state_dict=None):
    candidate = DynamicMLP(architecture["input_dim"], architecture["hidden_layers"], architecture["num_outputs"]).to(device)
    if state_dict is not None:
        candidate.load_state_dict(state_dict)
    return candidate


def state_dict_to_cpu(model):
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def make_loader(X, y, *, batch_size=EVAL_BATCH_SIZE, training=False):
    dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    generator = torch.Generator().manual_seed(SEED) if training else None
    drop_last = training and len(dataset) % batch_size == 1

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=training,
        drop_last=drop_last,
        generator=generator,
        pin_memory=device.type == "cuda",
    )


def predict_logits(model, data_loader):
    batches = []
    model.eval()
    with torch.inference_mode():
        for X, _ in data_loader:
            X = X.to(device, non_blocking=device.type == "cuda")
            batches.append(model(X).cpu())
    logits = torch.cat(batches).numpy()
    if not np.isfinite(logits).all():
        raise ValueError("Logit non finiti.")
    return logits


def precision_at_k(logits, targets, k=10):
    if logits.shape != targets.shape or not 1 <= k <= logits.shape[1]:
        raise ValueError(f"Shape/k non validi: logits={logits.shape}, targets={targets.shape}, k={k}")
    topk = np.argpartition(logits, logits.shape[1] - k, axis=1)[:, -k:]
    return float(np.take_along_axis(targets, topk, axis=1).sum(axis=1).mean() / k)


def mean_bce(logits, targets):
    loss = F.binary_cross_entropy_with_logits(torch.from_numpy(logits), torch.from_numpy(targets), reduction="none")
    return float(loss.mean(dim=1).mean())


def evaluate_loader(model, data_loader, targets):
    logits = predict_logits(model, data_loader)
    return {"logits": logits, "precision_at_10": precision_at_k(logits, targets), "bce": mean_bce(logits, targets)}


def logits_mse(a, b):
    if a.shape != b.shape:
        raise ValueError(f"Shape incompatibili: {a.shape} e {b.shape}")
    return float(np.mean(np.square(a - b)))


validation_loader = make_loader(X_validation, y_validation)
forget_loader = make_loader(X_forget, y_forget)

baseline_validation = evaluate_loader(model, validation_loader, y_validation)
baseline_forget = evaluate_loader(model, forget_loader, y_forget)
validation_logits = baseline_validation["logits"]
forget_logits = baseline_forget["logits"]
baseline_p10 = baseline_validation["precision_at_10"]
baseline_val_bce = baseline_validation["bce"]
baseline_forget_bce = baseline_forget["bce"]

baseline_results = pd.DataFrame([{
    "method": "baseline_original_model",
    "precision_at_10": baseline_p10,
    "validation_bce": baseline_val_bce,
    "forget_bce": baseline_forget_bce,
    "execution_time_seconds": 0.0,
    "best_epoch": np.nan,
}])

baseline_results_file = output_dir / "baseline_original_results.csv"
baseline_results.to_csv(baseline_results_file, index=False)
display(baseline_results)

## 9. Retraining da zero sul retain set

    Il retraining da zero e' il riferimento concettuale: un modello addestrato
    senza esempi forget. Non facciamo un secondo training includendo la validation,
    perche' `validation_ids.csv` deve rappresentare utenti esclusi dal training.

In [ ]:
import time

best_hyperparameters = dict(model_payload["best_hyperparameters"])
print("Iperparametri disponibili nell'artifact:")
print(best_hyperparameters)


def get_first_present(mapping, keys, *, required=False, default=None):
    for key in keys:
        if key in mapping:
            return mapping[key]
    if required:
        raise KeyError(f"Nessuna delle chiavi richieste e' presente: {keys}. Chiavi disponibili: {sorted(mapping)}")
    return default


LEARNING_RATE = float(get_first_present(best_hyperparameters, ("learning_rate", "lr"), required=True))
MAX_EPOCHS = int(get_first_present(best_hyperparameters, ("epochs", "num_epochs", "max_epochs"), required=True))
WEIGHT_DECAY = float(get_first_present(best_hyperparameters, ("weight_decay",), default=0.0))
TRAIN_BATCH_SIZE = int(get_first_present(best_hyperparameters, ("batch_size", "train_batch_size"), default=1024))
OPTIMIZER_NAME = str(get_first_present(best_hyperparameters, ("optimizer", "optimizer_name"), default="adam")).lower()
PATIENCE = 5

positive_counts = y_retain.sum(axis=0)
negative_counts = len(y_retain) - positive_counts
if np.any(positive_counts == 0):
    missing_targets = [target_cols[i] for i in np.flatnonzero(positive_counts == 0)]
    raise ValueError(f"Nel retain train alcune target non hanno esempi positivi: {missing_targets}")

pos_weight_values = negative_counts / (positive_counts + 1e-6)
pos_weight_values = np.clip(pos_weight_values, 0.1, 100.0)
pos_weight = torch.as_tensor(pos_weight_values, dtype=torch.float32, device=device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

training_assumptions = {
    "optimizer": OPTIMIZER_NAME,
    "loss": "BCEWithLogitsLoss",
    "pos_weight": "negative_count / positive_count, clipped to [0.1, 100.0]",
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "validation_excluded_from_training": True,
    "scheduler": "not available in artifact",
}

print("Architecture:", architecture)
print({
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "optimizer": OPTIMIZER_NAME,
    "patience": PATIENCE,
})
print("pos_weight min/mean/max:", f"{pos_weight.min().item():.3f}", f"{pos_weight.mean().item():.3f}", f"{pos_weight.max().item():.3f}")


def build_optimizer(model, *, optimizer_name, learning_rate, weight_decay):
    optimizer_name = optimizer_name.lower()
    if optimizer_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    if optimizer_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    if optimizer_name == "sgd":
        momentum = float(best_hyperparameters.get("momentum", 0.0))
        return torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay, momentum=momentum)
    raise ValueError(f"Optimizer non supportato: {optimizer_name!r}. Sono supportati: adam, adamw, sgd.")


def train_with_early_stopping(model, X_train, y_train, *, learning_rate, weight_decay, optimizer_name, train_batch_size, max_epochs, patience, criterion, validation_loader, y_validation):
    train_loader = make_loader(X_train, y_train, batch_size=train_batch_size, training=True)
    optimizer = build_optimizer(model, optimizer_name=optimizer_name, learning_rate=learning_rate, weight_decay=weight_decay)

    initial = evaluate_loader(model, validation_loader, y_validation)
    best_precision = initial["precision_at_10"]
    best_epoch = 0
    best_state = state_dict_to_cpu(model)
    epochs_without_improvement = 0
    history = []
    start_time = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = total_samples = 0

        for X, y in train_loader:
            X = X.to(device, non_blocking=device.type == "cuda")
            y = y.to(device, non_blocking=device.type == "cuda")
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X.size(0)
            total_samples += X.size(0)

        if total_samples == 0:
            raise RuntimeError("Il DataLoader di training non ha prodotto esempi.")

        metrics = evaluate_loader(model, validation_loader, y_validation)
        history.append({
            "epoch": epoch,
            "train_loss": total_loss / total_samples,
            "validation_bce": metrics["bce"],
            "validation_precision_at_10": metrics["precision_at_10"],
        })
        print(f"Epoch {epoch:03d} | train loss: {history[-1]['train_loss']:.6f} | validation BCE: {metrics['bce']:.6f} | P@10: {metrics['precision_at_10']:.6f}")

        if metrics["precision_at_10"] > best_precision:
            best_precision = metrics["precision_at_10"]
            best_epoch = epoch
            best_state = state_dict_to_cpu(model)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping all'epoca {epoch}. Migliore epoca: {best_epoch}.")
                break

    model.load_state_dict(best_state)
    model.eval()
    return {
        "model": model,
        "best_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_precision_at_10": best_precision,
        "execution_time_seconds": time.perf_counter() - start_time,
        "history": pd.DataFrame(history),
    }


torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

retrained_model = build_model()
retraining_result = train_with_early_stopping(
    retrained_model,
    X_retain,
    y_retain,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    optimizer_name=OPTIMIZER_NAME,
    train_batch_size=TRAIN_BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    criterion=criterion,
    validation_loader=validation_loader,
    y_validation=y_validation,
)

retrained_model = retraining_result["model"]
best_epoch = retraining_result["best_epoch"]
best_precision = retraining_result["best_precision_at_10"]
retraining_time = retraining_result["execution_time_seconds"]
training_history = retraining_result["history"]

retrained_validation = evaluate_loader(retrained_model, validation_loader, y_validation)
retrained_forget = evaluate_loader(retrained_model, forget_loader, y_forget)
retrained_p10 = retrained_validation["precision_at_10"]
retrained_val_bce = retrained_validation["bce"]
retrained_forget_bce = retrained_forget["bce"]

retrained_results = pd.DataFrame([{
    "method": "retraining_from_scratch",
    "precision_at_10": retrained_p10,
    "validation_bce": retrained_val_bce,
    "forget_bce": retrained_forget_bce,
    "execution_time_seconds": retraining_time,
    "best_epoch": best_epoch,
}])

comparison = pd.concat([baseline_results, retrained_results], ignore_index=True)
comparison_file = output_dir / "baseline_comparison.csv"
history_file = output_dir / "retraining_history.csv"
comparison.to_csv(comparison_file, index=False)
training_history.to_csv(history_file, index=False)

display(comparison)

Salviamo il retrained artifact nello stesso formato richiesto dalla challenge
    e lo ricarichiamo subito con `strict=True`, cosi' intercettiamo payload
    incompatibili prima della submission.

In [ ]:
retrained_artifact_file = output_dir / "model_artifact_retrained"
retrained_payload = {
    "state_dict": state_dict_to_cpu(retrained_model),
    "architecture": architecture,
    "best_hyperparameters": {
        **best_hyperparameters,
        "retraining_best_epoch": best_epoch,
        "retraining_validation_precision_at_10": best_precision,
        "retraining_time_seconds": retraining_time,
        "random_seed": SEED,
        "validation_size": VAL_SIZE,
    },
    "model_class_source": model_payload["model_class_source"],
    "training_assumptions": training_assumptions,
}

with open(retrained_artifact_file, "wb") as f:
    pickle.dump(retrained_payload, f)

with open(retrained_artifact_file, "rb") as f:
    loaded_payload = pickle.load(f)

test_model = DynamicMLP(
    loaded_payload["architecture"]["input_dim"],
    loaded_payload["architecture"]["hidden_layers"],
    loaded_payload["architecture"]["num_outputs"],
)
test_model.load_state_dict(loaded_payload["state_dict"])

print("Artifact retrained valido:", retrained_artifact_file)

## 10. Fine-tuning sul retain set

    Questa baseline parte dal modello originale e fa pochi aggiornamenti sul solo
    retain train. L'early stopping considera anche l'epoca 0: se il modello
    originale e' gia' migliore, non forziamo un fine-tuning peggiorativo.

In [ ]:
FINE_TUNING_LR = LEARNING_RATE * 0.1
FINE_TUNING_MAX_EPOCHS = min(MAX_EPOCHS, 20)
FINE_TUNING_PATIENCE = 3

fine_tuned_model = build_model(original_state_dict)
fine_tuning_result = train_with_early_stopping(
    fine_tuned_model,
    X_retain,
    y_retain,
    learning_rate=FINE_TUNING_LR,
    weight_decay=WEIGHT_DECAY,
    optimizer_name=OPTIMIZER_NAME,
    train_batch_size=TRAIN_BATCH_SIZE,
    max_epochs=FINE_TUNING_MAX_EPOCHS,
    patience=FINE_TUNING_PATIENCE,
    criterion=criterion,
    validation_loader=validation_loader,
    y_validation=y_validation,
)

fine_tuned_model = fine_tuning_result["model"]
fine_tuned_validation = evaluate_loader(fine_tuned_model, validation_loader, y_validation)
fine_tuned_forget = evaluate_loader(fine_tuned_model, forget_loader, y_forget)

experiment_results = pd.DataFrame([
    {
        "method": "original_model",
        "precision_at_10": baseline_p10,
        "validation_bce": baseline_val_bce,
        "forget_bce": baseline_forget_bce,
        "distance_to_retrained_validation": logits_mse(validation_logits, retrained_validation["logits"]),
        "distance_to_retrained_forget": logits_mse(forget_logits, retrained_forget["logits"]),
        "execution_time_seconds": 0.0,
        "best_epoch": np.nan,
    },
    {
        "method": "retraining_from_scratch",
        "precision_at_10": retrained_p10,
        "validation_bce": retrained_val_bce,
        "forget_bce": retrained_forget_bce,
        "distance_to_retrained_validation": 0.0,
        "distance_to_retrained_forget": 0.0,
        "execution_time_seconds": retraining_time,
        "best_epoch": best_epoch,
    },
    {
        "method": "retain_fine_tuning",
        "precision_at_10": fine_tuned_validation["precision_at_10"],
        "validation_bce": fine_tuned_validation["bce"],
        "forget_bce": fine_tuned_forget["bce"],
        "distance_to_retrained_validation": logits_mse(fine_tuned_validation["logits"], retrained_validation["logits"]),
        "distance_to_retrained_forget": logits_mse(fine_tuned_forget["logits"], retrained_forget["logits"]),
        "execution_time_seconds": fine_tuning_result["execution_time_seconds"],
        "best_epoch": fine_tuning_result["best_epoch"],
    },
])

experiment_results_file = output_dir / "experiment_comparison.csv"
fine_tuning_history_file = output_dir / "fine_tuning_history.csv"
experiment_results.to_csv(experiment_results_file, index=False)
fine_tuning_result["history"].to_csv(fine_tuning_history_file, index=False)

display(experiment_results)

Anche il modello fine-tuned viene salvato e verificato come artifact valido.

In [ ]:
fine_tuned_artifact_file = output_dir / "model_artifact_fine_tuned"
fine_tuned_payload = {
    "state_dict": state_dict_to_cpu(fine_tuned_model),
    "architecture": architecture,
    "best_hyperparameters": {
        **best_hyperparameters,
        "method": "retain_fine_tuning",
        "fine_tuning_learning_rate": FINE_TUNING_LR,
        "fine_tuning_best_epoch": fine_tuning_result["best_epoch"],
        "fine_tuning_validation_precision_at_10": fine_tuning_result["best_precision_at_10"],
        "fine_tuning_time_seconds": fine_tuning_result["execution_time_seconds"],
        "random_seed": SEED,
        "validation_size": VAL_SIZE,
    },
    "model_class_source": model_payload["model_class_source"],
    "training_assumptions": {**training_assumptions, "method": "retain_fine_tuning"},
}

with open(fine_tuned_artifact_file, "wb") as f:
    pickle.dump(fine_tuned_payload, f)

with open(fine_tuned_artifact_file, "rb") as f:
    loaded_fine_tuned_payload = pickle.load(f)

test_fine_tuned_model = DynamicMLP(
    loaded_fine_tuned_payload["architecture"]["input_dim"],
    loaded_fine_tuned_payload["architecture"]["hidden_layers"],
    loaded_fine_tuned_payload["architecture"]["num_outputs"],
)
test_fine_tuned_model.load_state_dict(loaded_fine_tuned_payload["state_dict"])

print("Artifact fine-tuned valido:", fine_tuned_artifact_file)

## 11. Strategia finale di machine unlearning

    La pipeline finale non si limita ad aumentare la loss sul forget set, perche'
    questo puo' distruggere utility e produrre logit instabili.

    ### 11.1 Fisher diagonale
    Calcoliamo Fisher empiriche separate su retain e forget. Il modello resta in
    `eval()` per impedire aggiornamenti accidentali delle statistiche BatchNorm.

    ### 11.2 Fisher ratio e selezione dei parametri
    Selezioniamo pesi con importanza relativa alta sul forget e importanza
    assoluta non trascurabile. I buffer BatchNorm non sono parametri ottimizzabili.

    ### 11.3 Selective Fisher Dampening
    Applichiamo dampening solo ai parametri selezionati, ripartendo sempre dai
    pesi originali per mantenere gli esperimenti indipendenti.

    ### 11.4 Repair fine-tuning
    Dopo il dampening ripariamo sul retain set per recuperare utility.

    ### 11.5 Knowledge distillation
    Distilliamo i logit del modello originale sul retain set per preservare il
    comportamento funzionale, non solo le label binarie.

    ### 11.6 Selective gradient ascent opzionale
    Il gradient ascent sul forget e' una variante controllata e mascherata: viene
    scelto solo se migliora realmente lo score locale.

## 12. Proxy locale per la privacy

    La proxy confronta il comportamento sul forget set del modello originale con
    quello del retrained. Serve a ordinare configurazioni, ma non sostituisce la
    MIA ufficiale nascosta.

    ## 13. Ricerca degli iperparametri

    Il runner prova configurazioni progressive di Fisher Dampening + repair e
    alcune varianti con gradient ascent. Il tempo della ricerca non viene incluso
    nel tempo finale della submission.

    ## 14. Selezione della configurazione finale

    La scelta finale bilancia utility, proxy privacy e tempo. Confrontiamo sempre
    il miglior metodo ibrido con il retraining da zero.

    ## 15. Generazione della submission

    L'ultima cella esegue il runner e genera `outputs/unlearning_final/submission/`
    con `model_artifact`, `execution_time.txt` e `validation_ids.csv`.

## 16. Verifica finale degli artifact

    Prima di concludere, il runner ricostruisce `DynamicMLP` e carica lo
    `state_dict` finale con `strict=True`. Questo controlla davvero che
    l'artifact sia utilizzabile dall'evaluator.

In [ ]:
summary = pd.DataFrame({
    "dataset": ["full_training", "retain_train", "validation", "forget"],
    "rows": [len(train_df), len(retain_train_df), len(validation_df), len(forget_df)],
    "replaced_feature_values": [repl_retain + repl_validation + repl_forget, repl_retain, repl_validation, repl_forget],
})

display(summary)
print(f"Feature: {len(feature_cols)} | Target: {len(target_cols)} | Device: {device}")
print(f"Train batch size: {TRAIN_BATCH_SIZE} | Eval batch size: {EVAL_BATCH_SIZE}")
print(f"Forget ID nel training: {len(forget_ids & train_ids)}/{len(forget_ids)}")
print(f"Validation IDs: {validation_ids_file}")
print(f"Baseline original: {baseline_results_file}")
print(f"Baseline comparison: {comparison_file}")
print(f"Experiment comparison: {experiment_results_file}")
print(f"Retraining history: {history_file}")
print(f"Fine-tuning history: {fine_tuning_history_file}")
print(f"Retrained artifact: {retrained_artifact_file}")
print(f"Fine-tuned artifact: {fine_tuned_artifact_file}")
print("Nota: validation utile per tuning/utilita', ma non necessariamente non-member per il modello originale.")

## 17. Risultati e conclusioni

    Dopo l'esecuzione completa, i CSV in `outputs/` raccontano baseline, ricerca,
    configurazione scelta e metriche locali. Il leaderboard resta la verifica
    definitiva per la MIA ufficiale, che non e' interamente pubblica.

In [ ]:
%run -i final_unlearning_runner.py